# RNA3D Phase 1 — DRfold2 reusable benchmark (Pilot + Batch)

**Run order:** mount → validate inputs → install pinned DRfold2 → preflight model → pilot → review → batch.

**Safety:** writes only to `RNA3D_Benchmark/DRfold2/runs/drfold2_pilot10_20261110_v1/`; it never edits NuFold, native structures, or previous DRfold2 smoke-test files.

**Experimental policy:** DRfold2 official default single-output inference (`python DRfold_infer.py input.fasta output_dir`). No native structures are passed into inference. Outputs are validated and archived per target. **TM-score/RMSD are not calculated here** until chain mapping, alignment coverage, and scoring conventions are reconciled with the NuFold evaluation pipeline (some references have missing residues).

**Colab runtime:** GPU (T4 recommended). Use **Pilot Mode** first. Batch Mode is opt-in and gated on a successful pilot.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, sys, json, csv, hashlib, shutil, subprocess, time, platform, tempfile, datetime
import torch

ROOT = Path('/content/drive/MyDrive/RNA3D_Benchmark')
NUFOLD = ROOT/'NuFold/runs/pilot10_single_20261018T203832Z'
INPUTS = NUFOLD/'inputs'
NATIVE = ROOT/'native/pilot_v1/native_eval'
TARGETS = ['8ubd_H','9fia_bJ','9cfn_B','9m31_B','9g4p_B',
           '9lq7_K','8z8u_B','9g6k_lB','9cf2_W','8wf8_B']
RUN_ID='drfold2_pilot10_20261110_v1'
RESULTS = ROOT/'DRfold2/runs'/RUN_ID
REPO = Path('/content/DRfold2')
COMMIT='3667d6ac44c5b14cff94c36e0371224acb2cdae5'
PILOT_TARGET='8ubd_H'
assert ROOT.is_dir() and INPUTS.is_dir() and NATIVE.is_dir(), 'Expected Drive folders are not mounted.'
assert torch.cuda.is_available(), 'Enable GPU in Runtime > Change runtime type, then rerun.'
print('GPU:',torch.cuda.get_device_name(0))
print('Input:', INPUTS)
print('Output (new, dedicated run):', RESULTS)


In [ ]:
def fasta_sequence(p):
    lines=[line.strip() for line in p.read_text().splitlines() if line.strip()]
    headers=[line for line in lines if line.startswith('>')]
    assert len(headers)==1 and lines[0].startswith('>'), f'Expected exactly one FASTA record: {p}'
    seq=''.join(line for line in lines if not line.startswith('>')).upper()
    assert seq and set(seq)<=set('ACGU'), f'Unexpected or empty sequence: {p}'
    return seq

def digest(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for part in iter(lambda:f.read(1024*1024), b''):
            h.update(part)
    return h.hexdigest()

manifest=[]
for target in TARGETS:
    fasta=sorted((INPUTS/target).glob('*.fasta'))
    assert len(fasta)==1, f'{target}: expected one FASTA, found {len(fasta)}'
    native=NATIVE/f'{target}_native_eval.pdb'
    assert native.is_file() and native.stat().st_size>100, f'{target}: native missing'
    seq=fasta_sequence(fasta[0])
    manifest.append(dict(target_id=target,fasta=str(fasta[0]),nucleotide_count=len(seq),
                         fasta_sha256=digest(fasta[0]),native=str(native),native_sha256=digest(native)))
print(f'Input integrity: {len(manifest)}/10 present with exactly one RNA FASTA + native PDB.')
for r in manifest: print(r['target_id'],r['nucleotide_count'],'nt')


## Install pinned upstream implementation

This checks/install clang++, downloads the official DRfold2 weights (~1.3 GB) as needed, and builds Arena **in ephemeral `/content`**, not in Drive. It may take substantial disk/network time. Failures stop execution.

The upstream inference script internally launches subprocesses; it can suppress error details. Therefore we verify required files, final PDB contents, and run logs, not merely its exit code. Official usage: https://github.com/leeyang/DRfold2


In [ ]:
def run_checked(args, cwd=None, timeout=None):
    print('$',' '.join(map(str,args)))
    subprocess.run([str(a) for a in args],cwd=cwd,check=True,timeout=timeout)

if not REPO.is_dir():
    run_checked(['git','clone','https://github.com/leeyang/DRfold2.git',REPO])
run_checked(['git','fetch','origin',COMMIT],cwd=REPO)
run_checked(['git','checkout','--detach',COMMIT],cwd=REPO)
current=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
assert current==COMMIT, (current,COMMIT)
print('Pinned upstream commit:',current)


In [ ]:
# Install required lightweight Python modules only if unavailable (do not replace Colab PyTorch).
import importlib.util
missing=[pkg for pkg in ['numpy','scipy','Bio'] if importlib.util.find_spec(pkg) is None]
if missing:
    run_checked([sys.executable,'-m','pip','install',*['biopython' if x=='Bio' else x for x in missing]])

# Colab fix verified on 2026-11-11: Arena's Makefile invokes clang++.
# Installing only gcc/g++ is insufficient and can fail after model download.
if shutil.which('clang++') is None:
    run_checked(['apt-get', 'update', '-qq'])
    run_checked(['apt-get', 'install', '-y', 'clang'])
assert shutil.which('clang++'), 'clang++ is required to compile Arena'

# Upstream dependencies and weights. Reuse an already installed model_hub if present.
# Installation script may download weights and compile Arena.
weights=REPO/'model_hub'
arena=REPO/'Arena/Arena'
weights_present = weights.is_dir() and all((weights/k).is_dir() and any((weights/k).iterdir()) for k in ['cfg_95','cfg_96','cfg_97','cfg_99'])
if weights_present and (REPO/'Arena/Makefile').is_file() and not arena.is_file():
    # Recover a partial install without downloading 1+ GB of weights again.
    run_checked(['make','Arena'],cwd=REPO/'Arena')
elif not (weights_present and arena.is_file()):
    run_checked(['bash','install.sh'],cwd=REPO)
assert arena.is_file() and os.access(arena,os.X_OK), 'Arena binary missing: inspect install output.'
for k in ['cfg_95','cfg_96','cfg_97','cfg_99']:
    q=weights/k
    assert q.is_dir() and any(q.iterdir()), f'Missing checkpoint folder: {q}'
print('Arena + four model checkpoint directories present.')
print('CUDA:',torch.version.cuda,'PyTorch:',torch.__version__)


## Inference engine

- Runs each target in `/content/rna3d_drfold2_work`, so intermediate files are not continuously streamed into Drive.
- Copies final `relax/model_1.pdb` and logs to a **new** target directory only when the structural check passes.
- Writes a `complete.json` marker **last** and verifies output hash before skipping previously finished targets.
- Keeps incomplete local artifacts for debugging in the current Colab session. On error, records a target-level failure.
- Does **not** delete, replace, or select native-derived best-of-N predictions.


In [ ]:
WORK=Path('/content/rna3d_drfold2_work')
WORK.mkdir(parents=True,exist_ok=True)

def verify_pdb(path):
    if not path.is_file() or path.stat().st_size<150:
        raise ValueError('Missing or tiny PDB')
    atoms=0; residues=set(); positions=set()
    for line in path.open(errors='replace'):
        if line.startswith(('ATOM  ','HETATM')):
            atoms+=1
            if len(line)>=54:
                residues.add((line[21],line[22:27]))
                for a,b in [(30,38),(38,46),(46,54)]:
                    float(line[a:b])
    if atoms<20 or len(residues)<3:
        raise ValueError(f'Invalid prediction PDB: atoms={atoms} residues={len(residues)}')
    return {'atom_records':atoms,'residue_count':len(residues),'size_bytes':path.stat().st_size}

def verified_complete(target):
    d=RESULTS/target
    marker=d/'complete.json'
    pdb=d/'model_1.pdb'
    if not marker.is_file() or not pdb.is_file(): return False
    try:
        data=json.loads(marker.read_text())
        return (data.get('status')=='complete' and data.get('target_id')==target
                and data.get('model_commit')==COMMIT
                and data.get('pdb_sha256')==digest(pdb)
                and bool(verify_pdb(pdb)))
    except Exception: return False

def persist_json_new(path,data):
    path.parent.mkdir(parents=True,exist_ok=True)
    # avoid overwriting immutable completion/result records
    with path.open('x') as f:
        json.dump(data,f,indent=2)

def execute_target(target):
    assert target in TARGETS
    if verified_complete(target):
        print('SKIP verified complete:',target)
        return 'skipped'
    dest=RESULTS/target
    if dest.exists():
        raise RuntimeError(f'{dest} already exists but lacks a verified complete marker; inspect manually. Nothing overwritten.')
    original=next(r for r in manifest if r['target_id']==target)
    assert digest(original['fasta'])==original['fasta_sha256'], 'Input changed since preflight'
    scratch=WORK/target
    if scratch.exists():
        raise RuntimeError(f'Local scratch exists: {scratch}; inspect before any retry.')
    scratch.mkdir(parents=True)
    log=scratch/'inference.log'
    command=[sys.executable,str(REPO/'DRfold_infer.py'),original['fasta'],str(scratch/'prediction')]
    started=time.time()
    try:
        print(f'START {target} ({original["nucleotide_count"]} nt)')
        with log.open('w') as f:
            p=subprocess.run(command,cwd=REPO,stdout=f,stderr=subprocess.STDOUT,check=False)
        final=scratch/'prediction/relax/model_1.pdb'
        if p.returncode:
            raise RuntimeError(f'Inference exit code {p.returncode}; inspect log.')
        details=verify_pdb(final)
        if details['residue_count'] != original['nucleotide_count']:
            raise ValueError(f"Prediction residue count mismatch: {details['residue_count']} vs {original['nucleotide_count']} nt; inspect PDB and atom mapping")
        meta={'status':'complete','target_id':target,'model':'DRfold2',
              'model_commit':COMMIT,'mode':'upstream_default_single_output',
              'fasta_sha256':original['fasta_sha256'],
              'native_sha256':original['native_sha256'],
              'pdb_sha256':digest(final),'seconds':round(time.time()-started,3),
              'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
              'command':command,'gpu':torch.cuda.get_device_name(0),
              'torch_version':torch.__version__,'pdb_validation':details}
        # Copy to a fresh result folder; record completion marker LAST.
        dest.mkdir(parents=True,exist_ok=False)
        shutil.copy2(final,dest/'model_1.pdb')
        shutil.copy2(log,dest/'inference.log')
        assert digest(dest/'model_1.pdb')==meta['pdb_sha256']
        persist_json_new(dest/'complete.json',meta)
        print('COMPLETE',target,'seconds=',meta['seconds'],'PDB residues=',details['residue_count'])
        return 'complete'
    except Exception as e:
        if not dest.exists():
            dest.mkdir(parents=True,exist_ok=False)
        if log.exists() and not (dest/'inference.log').exists():
            shutil.copy2(log,dest/'inference.log')
        persist_json_new(dest/'failed.json',{'target_id':target,'error':repr(e),
          'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()})
        print('FAILED',target,repr(e),'Inspect:',dest/'inference.log')
        raise


## PILOT MODE — one target only

Run this first. `8ubd_H` is short; its native evaluation PDB may contain fewer residues than the full input sequence, so **do not compute naive coordinate RMSD** against it.


In [ ]:
# Run the single-target pilot
pilot_status=execute_target(PILOT_TARGET)
print('Pilot status:',pilot_status)
assert verified_complete(PILOT_TARGET)
print('Pilot output:',RESULTS/PILOT_TARGET/'model_1.pdb')


In [ ]:
# Review pilot output and decide whether batch should be enabled.
p=RESULTS/PILOT_TARGET
print((p/'complete.json').read_text()[:3000])
print('Log tail:')
print('\n'.join((p/'inference.log').read_text(errors='replace').splitlines()[-25:]))
print('WARNING: no TM-score/RMSD calculated yet; map native residues first.')


## BATCH MODE — explicitly opt in after checking pilot

Change `RUN_BATCH = False` to `True` only after pilot output and logs look correct. The loop will skip previously verified complete targets. On any error, it stops to avoid wasted compute.

**Retry after a failed attempt:** do not erase the `failed.json` or output folder blindly; investigate the log, then use a new run ID or deliberately archive the failed folder first.


In [ ]:
RUN_BATCH = False  # Change to True only after pilot review
if not RUN_BATCH:
    print('Batch disabled. Pilot ready; review its structure and metadata first.')
else:
    assert verified_complete(PILOT_TARGET), 'Pilot must pass first.'
    for target in TARGETS:
        print('\n----',target,'----')
        execute_target(target)
    print('Verified complete:',sum(verified_complete(t) for t in TARGETS),'/',len(TARGETS))


## Read-only result overview

This cell only reads run artifacts and existing NuFold rank-1 results; it does not claim scored comparison. Final scoring requires standardized native/prediction residue mapping and a shared US-align configuration.


In [ ]:
print('DRfold2 outputs')
for t in TARGETS:
    d=RESULTS/t
    print(t, 'COMPLETE' if verified_complete(t) else ('FAILED / INCOMPLETE' if d.exists() else 'NOT RUN'))
nu=NUFOLD/'analysis/rank1_results.csv'
assert nu.exists(), 'NuFold rank-1 CSV missing'
with nu.open(newline='') as f:
    rows=list(csv.DictReader(f))
print('NuFold baseline scored target count:',sum(r['rank']=='1' and r['status']=='scored' for r in rows))
print('NEXT: validate chain mapping and run the same evaluation protocol on DRfold2 outputs.')
